In [ ]:
#batch1 SAVI阈值
import numpy as np
import matplotlib.pyplot as plt
import spectral as spy
import json
from shapely.geometry import Polygon, Point
import chardet

# === 1. 加载高光谱数据 ===
def load_hyperspectral_data(hdr_file):
    img = spy.open_image(hdr_file)
    data = np.array(img.load())
    if data.ndim == 3 and data.shape[0] < 50:  # (B,H,W) -> (H,W,B)
        data = np.transpose(data, (1, 2, 0))
    return data

# === 2. 自动检测 JSON 编码并读取 ===
def load_json_shapes(json_file):
    with open(json_file, 'rb') as f:
        raw_data = f.read()
    encoding = chardet.detect(raw_data)['encoding'] or 'utf-8'
    with open(json_file, 'r', encoding=encoding, errors='ignore') as f:
        annotations = json.load(f)
    return annotations['shapes']

# === 3. 提取标注区域像素 ===
def extract_pixels_from_shapes(data, shapes):
    pixels = []
    for shape in shapes:
        if shape['shape_type'] == 'polygon':
            points = shape['points']
            polygon = Polygon(points)
            for y in range(data.shape[0]):
                for x in range(data.shape[1]):
                    if polygon.contains(Point(x, y)):
                        pixels.append(data[y, x, :])
    return np.array(pixels)

# === 4. 计算 SAVI ===
def calculate_savi(pixels, nir_idx, red_idx, L=0.5):
    nir = pixels[:, nir_idx]
    red = pixels[:, red_idx]
    return ((nir - red) / (nir + red + L)) * (1 + L)

# === 5. 根据土壤样本用单尾 α=0.10 取临界值 ===
def calc_hypothesis_threshold(soil_savi_values, alpha=0.10):
    """
    输入：soil_savi_values = 土壤样本的 SAVI 值
    输出：临界阈值（分布左尾 alpha 分位点）
    """
    soil_savi_values = np.sort(soil_savi_values)
    threshold = np.percentile(soil_savi_values, alpha * 100)
    return threshold

# === 6. 绘制分布并标记阈值 ===
def plot_savi_histogram_with_threshold(savi_values, threshold, bins=200):
    counts, bin_edges = np.histogram(savi_values, bins=bins)
    plt.figure(figsize=(8, 5))
    plt.plot(bin_edges[:-1], counts, color='green', label='SAVI Histogram')
    plt.axvline(threshold, color='red', linestyle='--', label=f'α=0.10 Threshold={threshold:.4f}')
    plt.xlabel("SAVI 值")
    plt.ylabel("像素数量")
    plt.title("SAVI 分布及 α=0.10 单尾检验阈值")
    plt.legend()
    plt.grid(True)
    plt.show()

# === 7. 主程序 ===
def main():
    # === 文件路径和波段索引 ===
    hdr_file = r"H:\图像标记\0513\20250513nongdayancao_F1_2024-05-24_21-10-37-rect_refl.hdr"
    json_file = r"H:\图像标记\0513\20250513nongdayancao_F1_2024-05-24_21-10-37-rect_refl_hres.json"
    red_idx = 81
    nir_idx = 119

    print("1. 读取高光谱数据...")
    data = load_hyperspectral_data(hdr_file)
    print(f"数据维度: {data.shape}")

    print("2. 读取 JSON 标注...")
    shapes = load_json_shapes(json_file)
    print(f"标注区域数量: {len(shapes)}")

    print("3. 提取标注区域像素...")
    pixels = extract_pixels_from_shapes(data, shapes)
    print(f"提取像素数: {pixels.shape[0]}")

    if pixels.size == 0:
        print("⚠️ 没有提取到像素，请检查标注范围")
        return

    print("4. 计算 SAVI...")
    savi_values = calculate_savi(pixels, nir_idx, red_idx)

    # === 关键：假设 H0 = "像素属于土壤" ===
    print("5. 根据土壤样本计算 α=0.10 阈值...")
    threshold = calc_hypothesis_threshold(savi_values, alpha=0.10)
    print(f"α=0.10 单尾检验阈值 = {threshold:.4f}")

    print("6. 绘制直方图...")
    plot_savi_histogram_with_threshold(savi_values, threshold)

if __name__ == "__main__":
    main()


In [ ]:
#batch2 SAVI阈值
import numpy as np
import matplotlib.pyplot as plt
import spectral as spy
import json
from shapely.geometry import Polygon, Point
import chardet

# === 1. 加载高光谱数据 ===
def load_hyperspectral_data(hdr_file):
    img = spy.open_image(hdr_file)
    data = np.array(img.load())
    if data.ndim == 3 and data.shape[0] < 50:  # (B,H,W) -> (H,W,B)
        data = np.transpose(data, (1, 2, 0))
    return data

# === 2. 自动检测 JSON 编码并读取 ===
def load_json_shapes(json_file):
    with open(json_file, 'rb') as f:
        raw_data = f.read()
    encoding = chardet.detect(raw_data)['encoding'] or 'utf-8'
    with open(json_file, 'r', encoding=encoding, errors='ignore') as f:
        annotations = json.load(f)
    return annotations['shapes']

# === 3. 提取标注区域像素 ===
def extract_pixels_from_shapes(data, shapes):
    pixels = []
    for shape in shapes:
        if shape['shape_type'] == 'polygon':
            points = shape['points']
            polygon = Polygon(points)
            for y in range(data.shape[0]):
                for x in range(data.shape[1]):
                    if polygon.contains(Point(x, y)):
                        pixels.append(data[y, x, :])
    return np.array(pixels)

# === 4. 计算 SAVI ===
def calculate_savi(pixels, nir_idx, red_idx, L=0.5):
    nir = pixels[:, nir_idx]
    red = pixels[:, red_idx]
    return ((nir - red) / (nir + red + L)) * (1 + L)

# === 5. 根据土壤样本用单尾 α=0.10 取临界值 ===
def calc_hypothesis_threshold(soil_savi_values, alpha=0.10):
    """
    输入：soil_savi_values = 土壤样本的 SAVI 值
    输出：临界阈值（分布左尾 alpha 分位点）
    """
    soil_savi_values = np.sort(soil_savi_values)
    threshold = np.percentile(soil_savi_values, alpha * 100)
    return threshold

# === 6. 绘制分布并标记阈值 ===
def plot_savi_histogram_with_threshold(savi_values, threshold, bins=200):
    counts, bin_edges = np.histogram(savi_values, bins=bins)
    plt.figure(figsize=(8, 5))
    plt.plot(bin_edges[:-1], counts, color='green', label='SAVI Histogram')
    plt.axvline(threshold, color='red', linestyle='--', label=f'α=0.10 Threshold={threshold:.4f}')
    plt.xlabel("SAVI 值")
    plt.ylabel("像素数量")
    plt.title("SAVI 分布及 α=0.10 单尾检验阈值")
    plt.legend()
    plt.grid(True)
    plt.show()

# === 7. 主程序 ===
def main():
    # === 文件路径和波段索引 ===
    hdr_file = r"H:\图像标记\0627\20250627gaoqiaozaodao_F2_2024-05-24_23-52-13-rect_refl.hdr"
    json_file = r"H:\图像标记\0627\20250627gaoqiaozaodao_F2_2024-05-24_23-52-13-rect_refl_hres.json"
    red_idx = 81
    nir_idx = 119

    print("1. 读取高光谱数据...")
    data = load_hyperspectral_data(hdr_file)
    print(f"数据维度: {data.shape}")

    print("2. 读取 JSON 标注...")
    shapes = load_json_shapes(json_file)
    print(f"标注区域数量: {len(shapes)}")

    print("3. 提取标注区域像素...")
    pixels = extract_pixels_from_shapes(data, shapes)
    print(f"提取像素数: {pixels.shape[0]}")

    if pixels.size == 0:
        print("⚠️ 没有提取到像素，请检查标注范围")
        return

    print("4. 计算 SAVI...")
    savi_values = calculate_savi(pixels, nir_idx, red_idx)

    # === 关键：假设 H0 = "像素属于土壤" ===
    print("5. 根据土壤样本计算 α=0.10 阈值...")
    threshold = calc_hypothesis_threshold(savi_values, alpha=0.10)
    print(f"α=0.10 单尾检验阈值 = {threshold:.4f}")
    
    print("6. 绘制直方图...")
    plot_savi_histogram_with_threshold(savi_values, threshold)

if __name__ == "__main__":
    main()


In [ ]:
#batch3 SAVI阈值
import numpy as np
import matplotlib.pyplot as plt
import spectral as spy
import json
from shapely.geometry import Polygon, Point
import chardet

# === 1. 加载高光谱数据 ===
def load_hyperspectral_data(hdr_file):
    img = spy.open_image(hdr_file)
    data = np.array(img.load())
    if data.ndim == 3 and data.shape[0] < 50:  # (B,H,W) -> (H,W,B)
        data = np.transpose(data, (1, 2, 0))
    return data

# === 2. 自动检测 JSON 编码并读取 ===
def load_json_shapes(json_file):
    with open(json_file, 'rb') as f:
        raw_data = f.read()
    encoding = chardet.detect(raw_data)['encoding'] or 'utf-8'
    with open(json_file, 'r', encoding=encoding, errors='ignore') as f:
        annotations = json.load(f)
    return annotations['shapes']

# === 3. 提取标注区域像素 ===
def extract_pixels_from_shapes(data, shapes):
    pixels = []
    for shape in shapes:
        if shape['shape_type'] == 'polygon':
            points = shape['points']
            polygon = Polygon(points)
            for y in range(data.shape[0]):
                for x in range(data.shape[1]):
                    if polygon.contains(Point(x, y)):
                        pixels.append(data[y, x, :])
    return np.array(pixels)

# === 4. 计算 SAVI ===
def calculate_savi(pixels, nir_idx, red_idx, L=0.5):
    nir = pixels[:, nir_idx]
    red = pixels[:, red_idx]
    return ((nir - red) / (nir + red + L)) * (1 + L)

# === 5. 根据土壤样本用单尾 α=0.10 取临界值 ===
def calc_hypothesis_threshold(soil_savi_values, alpha=0.10):
    """
    输入：soil_savi_values = 土壤样本的 SAVI 值
    输出：临界阈值（分布左尾 alpha 分位点）
    """
    soil_savi_values = np.sort(soil_savi_values)
    threshold = np.percentile(soil_savi_values, alpha * 100)
    return threshold

# === 6. 绘制分布并标记阈值 ===
def plot_savi_histogram_with_threshold(savi_values, threshold, bins=200):
    counts, bin_edges = np.histogram(savi_values, bins=bins)
    plt.figure(figsize=(8, 5))
    plt.plot(bin_edges[:-1], counts, color='green', label='SAVI Histogram')
    plt.axvline(threshold, color='red', linestyle='--', label=f'α=0.10 Threshold={threshold:.4f}')
    plt.xlabel("SAVI 值")
    plt.ylabel("像素数量")
    plt.title("SAVI 分布及 α=0.10 单尾检验阈值")
    plt.legend()
    plt.grid(True)
    plt.show()

# === 7. 主程序 ===
def main():
    # === 文件路径和波段索引 ===
    hdr_file = r"H:\图像标记\0610\20250610nongdayancao_F2_2024-05-24_21-58-07-rect_refl.hdr"
    json_file = r"H:\图像标记\0610\20250610nongdayancao_F2_2024-05-24_21-58-07-rect_refl_hres.json"
    red_idx = 81
    nir_idx = 119

    print("1. 读取高光谱数据...")
    data = load_hyperspectral_data(hdr_file)
    print(f"数据维度: {data.shape}")

    print("2. 读取 JSON 标注...")
    shapes = load_json_shapes(json_file)
    print(f"标注区域数量: {len(shapes)}")

    print("3. 提取标注区域像素...")
    pixels = extract_pixels_from_shapes(data, shapes)
    print(f"提取像素数: {pixels.shape[0]}")

    if pixels.size == 0:
        print("⚠️ 没有提取到像素，请检查标注范围")
        return

    print("4. 计算 SAVI...")
    savi_values = calculate_savi(pixels, nir_idx, red_idx)

    # === 关键：假设 H0 = "像素属于土壤" ===
    print("5. 根据土壤样本计算 α=0.10 阈值...")
    threshold = calc_hypothesis_threshold(savi_values, alpha=0.10)
    print(f"α=0.10 单尾检验阈值 = {threshold:.4f}")
    

    print("6. 绘制直方图...")
    plot_savi_histogram_with_threshold(savi_values, threshold)

if __name__ == "__main__":
    main()


In [ ]:
# batch1 SAVI像素提取
import numpy as np
import matplotlib.pyplot as plt
import spectral as spy
import json
from shapely.geometry import Polygon, Point
import pandas as pd
from collections import Counter

# 1. 加载高光谱数据
def load_hyperspectral_data(hdr_file):
    img = spy.open_image(hdr_file)
    data = img.load()
    return data

# 2. 加载标记文件
def load_marking_data(json_file):
    with open(json_file, 'r') as f:
        markings = json.load(f)
    return markings

# 3. 使用 SAVI 指数进行筛选并提取植物像素
def map_markings_to_labeled_data_with_savi(
    data, markings, red_band_idx=81, nir_band_idx=119, savi_threshold=0.36, L=0.5
):
    labeled_pixels = []
    savi_values = []

    for shape in markings['shapes']:
        label = shape.get('label', 'unknown')
        points = shape['points']
        polygon = Polygon(points)

        for y in range(data.shape[0]):
            for x in range(data.shape[1]):
                if polygon.contains(Point(x, y)):
                    spectrum = data[y, x, :].squeeze()
                    red = float(spectrum[red_band_idx])
                    nir = float(spectrum[nir_band_idx])
                    savi = ((nir - red) / (nir + red + L)) * (1 + L)

                    savi_values.append(savi)

                    if savi > savi_threshold:
                        labeled_pixels.append((label, spectrum))

    # 输出 SAVI 分布
    if savi_values:
        print(f"SAVI min: {min(savi_values):.2f}, max: {max(savi_values):.2f}, mean: {np.mean(savi_values):.2f}")
    else:
        print("未计算到任何 SAVI 值")

    return labeled_pixels

# 4. 保存为 Excel 文件
def save_labeled_pixels_to_excel(labeled_pixels, save_path):
    if not labeled_pixels:
        print("没有提取到任何像素")
        return

    num_bands = labeled_pixels[0][1].shape[-1]
    column_names = ['Label'] + [f'Band_{i+1}' for i in range(num_bands)]

    rows = []
    for label, spectrum in labeled_pixels:
        spectrum = np.squeeze(spectrum)
        rows.append([label] + spectrum.tolist())

    df = pd.DataFrame(rows, columns=column_names)
    df.to_excel(save_path, index=False)
    print(f"Excel 文件已保存到：{save_path}")

# 5. 可视化所有筛选后的光谱曲线
def visualize_all_labeled_pixels(labeled_pixels):
    plt.figure(figsize=(14, 8))
    seen_labels = set()

    for label, spectrum in labeled_pixels:
        label_for_legend = label if label not in seen_labels else ""
        plt.plot(np.squeeze(spectrum), alpha=0.4, label=label_for_legend)
        seen_labels.add(label)

    plt.xlabel('Band Index')
    plt.ylabel('Reflectance')
    plt.title('SAVI-Filtered Labeled Pixels')
    plt.legend()
    plt.tight_layout()
    plt.show()

# 6. 主函数
def main():
    hdr_file = r"H:\图像标记\0513\20250513nongdayancao_F1_2024-05-24_21-10-37-rect_refl.hdr"
    json_file = r"H:\图像标记\0513\20250513nongdayancao_F1_2024-05-24_21-10-37-rect_refl_hres.json"
    xlsx_file = r"H:\图像标记\0513\0.36SAVI.xlsx"

    # 波段索引（已确定）
    red_band_index = 81   # ~670nm
    nir_band_index = 119  # ~800nm

    savi_threshold = 0.36  # 使用指定阈值

    print("开始加载数据...")
    hyperspectral_data = load_hyperspectral_data(hdr_file)
    markings = load_marking_data(json_file)

    print("提取标注区域内的像素并应用 SAVI 筛选...")
    labeled_pixels = map_markings_to_labeled_data_with_savi(
        hyperspectral_data,
        markings,
        red_band_idx=red_band_index,
        nir_band_idx=nir_band_index,
        savi_threshold=savi_threshold
    )

    print(f"共提取植物像素数（SAVI > {savi_threshold}）：{len(labeled_pixels)}")

    # 新增：统计每类标签像素数量
    label_counts = Counter([label for label, _ in labeled_pixels])
    if label_counts:
        print("各标签筛选后的像素数量：")
        for label, count in label_counts.items():
            print(f"  {label}: {count}")
    else:
        print("没有提取到任何像素，无法统计标签数量。")

    # 保存 Excel 文件
    save_labeled_pixels_to_excel(labeled_pixels, xlsx_file)

    # 可视化光谱
    visualize_all_labeled_pixels(labeled_pixels)

if __name__ == "__main__":
    main()


In [ ]:
#batch2 SAVI像素提取
import numpy as np
import matplotlib.pyplot as plt
import spectral as spy
import json
from shapely.geometry import Polygon, Point
import pandas as pd
from collections import Counter

# 1. 加载高光谱数据
def load_hyperspectral_data(hdr_file):
    img = spy.open_image(hdr_file)
    data = img.load()
    return data

# 2. 加载标记文件
def load_marking_data(json_file):
    with open(json_file, 'r') as f:
        markings = json.load(f)
    return markings

# 3. 使用 SAVI 指数进行筛选并提取植物像素
def map_markings_to_labeled_data_with_savi(
    data, markings, red_band_idx=81, nir_band_idx=119, savi_threshold=0.47, L=0.5
):
    labeled_pixels = []
    savi_values = []

    for shape in markings['shapes']:
        label = shape.get('label', 'unknown')
        points = shape['points']
        polygon = Polygon(points)

        for y in range(data.shape[0]):
            for x in range(data.shape[1]):
                if polygon.contains(Point(x, y)):
                    spectrum = data[y, x, :].squeeze()
                    red = float(spectrum[red_band_idx])
                    nir = float(spectrum[nir_band_idx])
                    savi = ((nir - red) / (nir + red + L)) * (1 + L)

                    savi_values.append(savi)

                    if savi > savi_threshold:
                        labeled_pixels.append((label, spectrum))

    # 输出 SAVI 分布
    if savi_values:
        print(f"SAVI min: {min(savi_values):.2f}, max: {max(savi_values):.2f}, mean: {np.mean(savi_values):.2f}")
    else:
        print("未计算到任何 SAVI 值")

    return labeled_pixels

# 4. 保存为 Excel 文件
def save_labeled_pixels_to_excel(labeled_pixels, save_path):
    if not labeled_pixels:
        print("没有提取到任何像素")
        return

    num_bands = labeled_pixels[0][1].shape[-1]
    column_names = ['Label'] + [f'Band_{i+1}' for i in range(num_bands)]

    rows = []
    for label, spectrum in labeled_pixels:
        spectrum = np.squeeze(spectrum)
        rows.append([label] + spectrum.tolist())

    df = pd.DataFrame(rows, columns=column_names)
    df.to_excel(save_path, index=False)
    print(f"Excel 文件已保存到：{save_path}")

# 5. 可视化所有筛选后的光谱曲线
def visualize_all_labeled_pixels(labeled_pixels):
    plt.figure(figsize=(14, 8))
    seen_labels = set()

    for label, spectrum in labeled_pixels:
        label_for_legend = label if label not in seen_labels else ""
        plt.plot(np.squeeze(spectrum), alpha=0.4, label=label_for_legend)
        seen_labels.add(label)

    plt.xlabel('Band Index')
    plt.ylabel('Reflectance')
    plt.title('SAVI-Filtered Labeled Pixels')
    plt.legend()
    plt.tight_layout()
    plt.show()

# 6. 主函数
def main():
    hdr_file = r"H:\图像标记\0627\20250627gaoqiaozaodao_F2_2024-05-24_23-52-13-rect_refl.hdr"
    json_file = r"H:\图像标记\0627\20250627gaoqiaozaodao_F2_2024-05-24_23-52-13-rect_refl_hres.json"
    xlsx_file = r"H:\图像标记\0627\0.47SAVI.xlsx"

    # 波段索引（已确认）
    red_band_index = 81   # ~670nm
    nir_band_index = 119  # ~800nm

    savi_threshold = 0.47  # 使用新的阈值

    print("开始加载数据...")
    hyperspectral_data = load_hyperspectral_data(hdr_file)
    markings = load_marking_data(json_file)

    print("提取标注区域内的像素并应用 SAVI 筛选...")
    labeled_pixels = map_markings_to_labeled_data_with_savi(
        hyperspectral_data,
        markings,
        red_band_idx=red_band_index,
        nir_band_idx=nir_band_index,
        savi_threshold=savi_threshold
    )

    print(f"共提取植物像素数（SAVI > {savi_threshold}）：{len(labeled_pixels)}")

    # 新增：统计每类标签像素数量
    label_counts = Counter([label for label, _ in labeled_pixels])
    if label_counts:
        print("各标签筛选后的像素数量：")
        for label, count in label_counts.items():
            print(f"  {label}: {count}")
    else:
        print("没有提取到任何像素，无法统计标签数量。")

    # 保存 Excel 文件
    save_labeled_pixels_to_excel(labeled_pixels, xlsx_file)

    # 可视化光谱
    visualize_all_labeled_pixels(labeled_pixels)

if __name__ == "__main__":
    main()


In [ ]:
#batch3 SAVI像素提取
import numpy as np
import matplotlib.pyplot as plt
import spectral as spy
import json
from shapely.geometry import Polygon, Point
import pandas as pd
from collections import Counter

# 1. 加载高光谱数据
def load_hyperspectral_data(hdr_file):
    img = spy.open_image(hdr_file)
    data = img.load()
    return data

# 2. 加载标记文件
def load_marking_data(json_file):
    with open(json_file, 'r') as f:
        markings = json.load(f)
    return markings

# 3. 使用 SAVI 指数进行筛选并提取植物像素
def map_markings_to_labeled_data_with_savi(
    data, markings, red_band_idx=81, nir_band_idx=119, savi_threshold=0.52, L=0.5
):
    labeled_pixels = []
    savi_values = []

    for shape in markings['shapes']:
        label = shape.get('label', 'unknown')
        points = shape['points']
        polygon = Polygon(points)

        for y in range(data.shape[0]):
            for x in range(data.shape[1]):
                if polygon.contains(Point(x, y)):
                    spectrum = data[y, x, :].squeeze()
                    red = float(spectrum[red_band_idx])
                    nir = float(spectrum[nir_band_idx])
                    savi = ((nir - red) / (nir + red + L)) * (1 + L)

                    savi_values.append(savi)

                    if savi > savi_threshold:
                        labeled_pixels.append((label, spectrum))

    # 输出 SAVI 分布
    if savi_values:
        print(f"SAVI min: {min(savi_values):.2f}, max: {max(savi_values):.2f}, mean: {np.mean(savi_values):.2f}")
    else:
        print("未计算到任何 SAVI 值")

    return labeled_pixels

# 4. 保存为 Excel 文件
def save_labeled_pixels_to_excel(labeled_pixels, save_path):
    if not labeled_pixels:
        print("没有提取到任何像素")
        return

    num_bands = labeled_pixels[0][1].shape[-1]
    column_names = ['Label'] + [f'Band_{i+1}' for i in range(num_bands)]

    rows = []
    for label, spectrum in labeled_pixels:
        spectrum = np.squeeze(spectrum)
        rows.append([label] + spectrum.tolist())

    df = pd.DataFrame(rows, columns=column_names)
    df.to_excel(save_path, index=False)
    print(f"Excel 文件已保存到：{save_path}")

# 5. 可视化所有筛选后的光谱曲线
def visualize_all_labeled_pixels(labeled_pixels):
    plt.figure(figsize=(14, 8))
    seen_labels = set()

    for label, spectrum in labeled_pixels:
        label_for_legend = label if label not in seen_labels else ""
        plt.plot(np.squeeze(spectrum), alpha=0.4, label=label_for_legend)
        seen_labels.add(label)

    plt.xlabel('Band Index')
    plt.ylabel('Reflectance')
    plt.title('SAVI-Filtered Labeled Pixels')
    plt.legend()
    plt.tight_layout()
    plt.show()

# 6. 主函数
def main():
    hdr_file = r"H:\图像标记\0610\20250610nongdayancao_F2_2024-05-24_21-58-07-rect_refl.hdr"
    json_file = r"H:\图像标记\0610\20250610nongdayancao_F2_2024-05-24_21-58-07-rect_refl_hres.json"
    xlsx_file = r"H:\图像标记\0610\0.52SAVI.xlsx"

    # 波段索引（已确定）
    red_band_index = 81   # ~670nm
    nir_band_index = 119  # ~800nm

    savi_threshold = 0.52  # 使用指定阈值

    print("开始加载数据...")
    hyperspectral_data = load_hyperspectral_data(hdr_file)
    markings = load_marking_data(json_file)

    print("提取标注区域内的像素并应用 SAVI 筛选...")
    labeled_pixels = map_markings_to_labeled_data_with_savi(
        hyperspectral_data,
        markings,
        red_band_idx=red_band_index,
        nir_band_idx=nir_band_index,
        savi_threshold=savi_threshold
    )

    print(f"共提取植物像素数（SAVI > {savi_threshold}）：{len(labeled_pixels)}")

    # 新增：统计每类标签像素数量
    label_counts = Counter([label for label, _ in labeled_pixels])
    if label_counts:
        print("各标签筛选后的像素数量：")
        for label, count in label_counts.items():
            print(f"  {label}: {count}")
    else:
        print("没有提取到任何像素，无法统计标签数量。")

    # 保存 Excel 文件
    save_labeled_pixels_to_excel(labeled_pixels, xlsx_file)

    # 可视化光谱
    visualize_all_labeled_pixels(labeled_pixels)

if __name__ == "__main__":
    main()


In [ ]:
#将第一批数据和第二批数据合并为训练集
import pandas as pd

# 文件路径
file1 = r"H:\图像标记\0513\0.36SAVI.xlsx"
file2 = r"H:\图像标记\0627\0.47SAVI.xlsx"
output_file = r"H:\图像标记\合并结果_513_627.xlsx"

# 读取文件
df1 = pd.read_excel(file1)
df2 = pd.read_excel(file2)

# 检查列名一致性，重新对齐
columns = list(df1.columns)
df2 = df2.reindex(columns=columns)

# 合并结果列表
merged_list = []

# 假设标签是第一列
label_col = columns[0]
labels = sorted(df1[label_col].unique())

for label in labels:
    # 分别取出两个文件中对应标签的行
    part1 = df1[df1[label_col] == label]
    part2 = df2[df2[label_col] == label]
    
    # 按顺序加入列表
    merged_list.append(part1)
    merged_list.append(part2)

# 合并所有部分
merged_df = pd.concat(merged_list, ignore_index=True)

# 保存结果
merged_df.to_excel(output_file, index=False)
print(f"合并完成，已保存到 {output_file}")
